In [1]:
import os
import GEOparse
import requests
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[sample_title] = metadata_object


# Create metadata DataFrame and join as columns to df_genes (rows=samples)
metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'

# Use author provided expression data


In [2]:
url = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE100797&format=file&file=GSE100797%5FProcessedData%2Etxt%2Egz"
output_gz = "GSE100797_ProcessedData.txt.gz"

# Download the gene expression data.
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open("GSE100797_ProcessedData.txt.gz", "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

In [3]:
df_genes = pd.read_csv('GSE100797_ProcessedData.txt.gz', sep='\t')
df_genes.rename(columns={'Unnamed: 0': 'gene_names'}, inplace=True)
df_genes.set_index('gene_names', inplace=True)
df_genes = df_genes.transpose()
df_genes.index.name = 'samples'
df_genes.columns.name = None


data_file_name = f'{GEO_ID}_original.csv'
df_combined = metadata_df.join(df_genes, how='inner')
df_combined.reset_index(inplace=True)
df_combined.to_csv(data_file_name, index=False)
df_combined

,samples,sample.timepoint,ajcc.stage,pfs.time,pfs.event,os.time,os.event,tumor.response,type.of.lesion,type.of.primary,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
0,MM909_01_1,Before TILs,M1c,48.1,1,92.9,0,-100,LN,unknown,...,2.20174,2.70930,0.82517,1.36987,2.76248,0.19923,2.55180,5.53388,2.56353,3.15532
1,MM909_02_1,Before TILs,M1b,2,1,6.9,1,10.94,LN,skin,...,3.40559,2.97088,1.27695,1.29983,2.69095,0.02866,2.45870,5.42459,1.80400,3.68111
2,MM909_03_1,Before TILs,M1c,3.8,1,11.4,1,11.92,SC,unknown,...,2.79518,3.91229,1.19076,2.07418,2.84976,0.11189,3.43125,5.08218,2.69648,4.74777
3,MM909_06_1,Before TILs,M1c,2.3,1,4.6,1,-7.14,LN,skin,...,3.22395,4.48321,0.38875,0.91491,1.91299,0.05874,2.96821,5.00434,0.54810,3.79871
4,MM909_11_1,Before TILs,M1a,13.1,1,71.5,0,-100,LN,skin,...,3.54445,3.98488,0.71845,1.15190,2.64073,0.06551,2.92381,5.07525,2.02028,3.90744
5,MM909_14_1,Before TILs,M1c,2.5,1,3.2,1,27.8,LN,unknown,...,3.08617,5.31298,0.72228,0.93854,2.20358,0.07093,1.98342,6.21047,2.56292,3.58276
6,MM909_15_2,Before TILs,M1b,67.13,0,67.1,0,-100,LN,skin,...,2.92596,4.16607,0.91999,1.66906,2.76124,0.00000,2.89690,5.03869,1.10683,4.24981
7,MM909_16_1,Before TILs,M1c,3.9,1,65.3,0,-12,SC,skin,...,3.28836,3.66968,1.13496,1.73852,2.57539,0.09963,2.91116,4.58243,2.39091,4.52282
8,MM909_17_1,Before TILs,M1c,64.87,0,64.9,0,-40.5,LN,skin,...,2.55088,3.75524,1.17525,1.78707,2.46269,0.28640,3.84822,3.50055,1.05258,4.66200
9,MM909_22_2,After Ipilimumab/Before TILs,M1c,32.1,0,32.1,1,-100,NA,skin,...,2.42877,3.13928,0.83870,1.01080,2.53381,0.07743,2.96283,7.09065,2.47918,3.40053


# Generate description

In [4]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)
    print("Description generated")

desc_data['data_summary'] = {
    'samples': metadata_df.reset_index().shape[0],
    'clinical_features': metadata_df.reset_index().shape[1],
    'genes': df_genes.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
